In [1]:
import pyspark
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("Miniproject1").getOrCreate()

In [3]:
spark

In [4]:
import os
import shutil

BASE_DIR = os.getcwd()
input_path = os.path.join(BASE_DIR, "input")
error_path = os.path.join(BASE_DIR, "error")

# Create folders if they don't exist
for p in [input_path, error_path]:
    os.makedirs(p, exist_ok=True)

# Source CSV file (adjust this if file name changes)
source_file = os.path.join(BASE_DIR, "data.csv")

# Destination path inside input folder
destination_file = os.path.join(input_path, "data.csv")

# Move file only if it exists
if os.path.exists(source_file):
    shutil.move(source_file, destination_file)
    print("data.csv moved to input folder.")
else:
    print("data.csv not found in base directory.")


data.csv not found in base directory.


In [5]:
read_csv = spark.read.csv(destination_file, header=True, inferSchema=True)
read_csv.show()

+----------+--------------------+-------------------+----------+--------+
|     Stock|              Sector|              Month|PriceStart|PriceEnd|
+----------+--------------------+-------------------+----------+--------+
|      AAPL|          Technology|2025-01-01 00:00:00|     120.5|   175.2|
|      MSFT|          Technology|2025-02-01 00:00:00|     210.1|   280.5|
|       WMT|    Consumer Staples|2025-03-01 00:00:00|      NULL|   150.0|
|      TSLA|          Automotive|2025-04-01 00:00:00|       0.0|     0.0|
|       JPM|          Financials|2025-05-01 00:00:00|      95.0|   105.0|
|      BABA|Consumer Discreti...|2025-06-01 00:00:00|     180.0|   160.0|
|  RELIANCE|              Energy|2025-07-01 00:00:00|    2200.0|  2400.0|
|       TCS|          Technology|2025-08-01 00:00:00|    3100.0|  3500.0|
|      HDFC|          Financials|2025-09-01 00:00:00|    1500.0|  3650.0|
|BAJAJ-AUTO|          Automotive|2025-10-01 00:00:00|    4800.0|  7200.0|
+----------+--------------------+-----

In [6]:
read_csv.printSchema()

root
 |-- Stock: string (nullable = true)
 |-- Sector: string (nullable = true)
 |-- Month: timestamp (nullable = true)
 |-- PriceStart: double (nullable = true)
 |-- PriceEnd: double (nullable = true)



In [7]:
from pyspark.sql.functions import col, when, expr

# Step 1: Treat columns as strings and normalize malformed text to NULL
df_step1 = read_csv.withColumn("PriceStart", col("PriceStart").cast("string")) \
                   .withColumn("PriceStart", when(col("PriceStart").isin("NULL", "", "NaN"), None).otherwise(col("PriceStart"))) \
                   .withColumn("PriceEnd", col("PriceEnd").cast("string")) \
                   .withColumn("PriceEnd", when(col("PriceEnd").isin("NULL", "", "NaN"), None).otherwise(col("PriceEnd")))

# Step 2: Safe cast using try_cast (returns NULL for malformed values)
df_step2 = df_step1.select(
    expr("try_cast(PriceStart as double)").alias("PriceStart"),
    expr("try_cast(PriceEnd as double)").alias("PriceEnd"),
    *[c for c in read_csv.columns if c not in ("PriceStart", "PriceEnd")]
)

# Step 3: Replace NULLs with 0 (or another sentinel if preferred)
df_clean = df_step2.na.fill({"PriceStart": 0, "PriceEnd": 0})

df_clean.show()
df_clean.printSchema()


+----------+--------+----------+--------------------+-------------------+
|PriceStart|PriceEnd|     Stock|              Sector|              Month|
+----------+--------+----------+--------------------+-------------------+
|     120.5|   175.2|      AAPL|          Technology|2025-01-01 00:00:00|
|     210.1|   280.5|      MSFT|          Technology|2025-02-01 00:00:00|
|       0.0|   150.0|       WMT|    Consumer Staples|2025-03-01 00:00:00|
|       0.0|     0.0|      TSLA|          Automotive|2025-04-01 00:00:00|
|      95.0|   105.0|       JPM|          Financials|2025-05-01 00:00:00|
|     180.0|   160.0|      BABA|Consumer Discreti...|2025-06-01 00:00:00|
|    2200.0|  2400.0|  RELIANCE|              Energy|2025-07-01 00:00:00|
|    3100.0|  3500.0|       TCS|          Technology|2025-08-01 00:00:00|
|    1500.0|  3650.0|      HDFC|          Financials|2025-09-01 00:00:00|
|    4800.0|  7200.0|BAJAJ-AUTO|          Automotive|2025-10-01 00:00:00|
+----------+--------+----------+------

In [8]:
from pyspark.sql.functions import col
import time
invalid_df = read_csv.filter((col("PriceStart") <= 0) | (col("PriceEnd") <= 0))

#if invalid_df.count() > 0:
   #invalid_df.write \
    #.mode("overwrite") \
    #.option("header", True) \
    #.csv(rf"C:\Users\RakshithM\PycharmProjects\PythonProject\Miniprojects\error\Bad_Data_{timestamp}.csv")


valid_data = df_clean.filter((col("PriceStart") > 0) & (col("PriceEnd") > 0))
valid_data.show()


+----------+--------+----------+--------------------+-------------------+
|PriceStart|PriceEnd|     Stock|              Sector|              Month|
+----------+--------+----------+--------------------+-------------------+
|     120.5|   175.2|      AAPL|          Technology|2025-01-01 00:00:00|
|     210.1|   280.5|      MSFT|          Technology|2025-02-01 00:00:00|
|      95.0|   105.0|       JPM|          Financials|2025-05-01 00:00:00|
|     180.0|   160.0|      BABA|Consumer Discreti...|2025-06-01 00:00:00|
|    2200.0|  2400.0|  RELIANCE|              Energy|2025-07-01 00:00:00|
|    3100.0|  3500.0|       TCS|          Technology|2025-08-01 00:00:00|
|    1500.0|  3650.0|      HDFC|          Financials|2025-09-01 00:00:00|
|    4800.0|  7200.0|BAJAJ-AUTO|          Automotive|2025-10-01 00:00:00|
+----------+--------+----------+--------------------+-------------------+



In [9]:
from pyspark.sql.functions import col
Price_movement =valid_data.withColumn("PriceChange", col("PriceEnd")-col("PriceStart"))
Price_movement.show()

+----------+--------+----------+--------------------+-------------------+-----------------+
|PriceStart|PriceEnd|     Stock|              Sector|              Month|      PriceChange|
+----------+--------+----------+--------------------+-------------------+-----------------+
|     120.5|   175.2|      AAPL|          Technology|2025-01-01 00:00:00|54.69999999999999|
|     210.1|   280.5|      MSFT|          Technology|2025-02-01 00:00:00|             70.4|
|      95.0|   105.0|       JPM|          Financials|2025-05-01 00:00:00|             10.0|
|     180.0|   160.0|      BABA|Consumer Discreti...|2025-06-01 00:00:00|            -20.0|
|    2200.0|  2400.0|  RELIANCE|              Energy|2025-07-01 00:00:00|            200.0|
|    3100.0|  3500.0|       TCS|          Technology|2025-08-01 00:00:00|            400.0|
|    1500.0|  3650.0|      HDFC|          Financials|2025-09-01 00:00:00|           2150.0|
|    4800.0|  7200.0|BAJAJ-AUTO|          Automotive|2025-10-01 00:00:00|       

In [12]:
from pyspark.sql.functions import when
GainLoss = Price_movement.withColumn("catogory", when(col("PriceEnd") > col("PriceStart"), "Gain").otherwise("Loss"))
GainLoss.show()

+----------+--------+----------+--------------------+-------------------+-----------------+--------+
|PriceStart|PriceEnd|     Stock|              Sector|              Month|      PriceChange|catogory|
+----------+--------+----------+--------------------+-------------------+-----------------+--------+
|     120.5|   175.2|      AAPL|          Technology|2025-01-01 00:00:00|54.69999999999999|    Gain|
|     210.1|   280.5|      MSFT|          Technology|2025-02-01 00:00:00|             70.4|    Gain|
|      95.0|   105.0|       JPM|          Financials|2025-05-01 00:00:00|             10.0|    Gain|
|     180.0|   160.0|      BABA|Consumer Discreti...|2025-06-01 00:00:00|            -20.0|    Loss|
|    2200.0|  2400.0|  RELIANCE|              Energy|2025-07-01 00:00:00|            200.0|    Gain|
|    3100.0|  3500.0|       TCS|          Technology|2025-08-01 00:00:00|            400.0|    Gain|
|    1500.0|  3650.0|      HDFC|          Financials|2025-09-01 00:00:00|           2150.0|

In [15]:
from pyspark.sql.functions import round
Returns = GainLoss.withColumn("Return%" , round((col("PriceEnd")-col("PriceStart"))/col("PriceStart")*100 , 2))
Returns.show()

+----------+--------+----------+--------------------+-------------------+-----------------+--------+-------+
|PriceStart|PriceEnd|     Stock|              Sector|              Month|      PriceChange|catogory|Return%|
+----------+--------+----------+--------------------+-------------------+-----------------+--------+-------+
|     120.5|   175.2|      AAPL|          Technology|2025-01-01 00:00:00|54.69999999999999|    Gain|  45.39|
|     210.1|   280.5|      MSFT|          Technology|2025-02-01 00:00:00|             70.4|    Gain|  33.51|
|      95.0|   105.0|       JPM|          Financials|2025-05-01 00:00:00|             10.0|    Gain|  10.53|
|     180.0|   160.0|      BABA|Consumer Discreti...|2025-06-01 00:00:00|            -20.0|    Loss| -11.11|
|    2200.0|  2400.0|  RELIANCE|              Energy|2025-07-01 00:00:00|            200.0|    Gain|   9.09|
|    3100.0|  3500.0|       TCS|          Technology|2025-08-01 00:00:00|            400.0|    Gain|   12.9|
|    1500.0|  3650.

In [16]:
from pyspark.sql.functions import col, when
sort_data1 = Returns.withColumn("StocksPerformance", when(col("Return%") > 20, "Excellent").when((col("Return%") > 5) & (col("Return%") <= 20), "Good").otherwise("Poor"))
sort_data1.show()

+----------+--------+----------+--------------------+-------------------+-----------------+--------+-------+-----------------+
|PriceStart|PriceEnd|     Stock|              Sector|              Month|      PriceChange|catogory|Return%|StocksPerformance|
+----------+--------+----------+--------------------+-------------------+-----------------+--------+-------+-----------------+
|     120.5|   175.2|      AAPL|          Technology|2025-01-01 00:00:00|54.69999999999999|    Gain|  45.39|        Excellent|
|     210.1|   280.5|      MSFT|          Technology|2025-02-01 00:00:00|             70.4|    Gain|  33.51|        Excellent|
|      95.0|   105.0|       JPM|          Financials|2025-05-01 00:00:00|             10.0|    Gain|  10.53|             Good|
|     180.0|   160.0|      BABA|Consumer Discreti...|2025-06-01 00:00:00|            -20.0|    Loss| -11.11|             Poor|
|    2200.0|  2400.0|  RELIANCE|              Energy|2025-07-01 00:00:00|            200.0|    Gain|   9.09|   

In [17]:
sort_data = Returns.orderBy("Return%")
sort_data.show()

+----------+--------+----------+--------------------+-------------------+-----------------+--------+-------+
|PriceStart|PriceEnd|     Stock|              Sector|              Month|      PriceChange|catogory|Return%|
+----------+--------+----------+--------------------+-------------------+-----------------+--------+-------+
|     180.0|   160.0|      BABA|Consumer Discreti...|2025-06-01 00:00:00|            -20.0|    Loss| -11.11|
|    2200.0|  2400.0|  RELIANCE|              Energy|2025-07-01 00:00:00|            200.0|    Gain|   9.09|
|      95.0|   105.0|       JPM|          Financials|2025-05-01 00:00:00|             10.0|    Gain|  10.53|
|    3100.0|  3500.0|       TCS|          Technology|2025-08-01 00:00:00|            400.0|    Gain|   12.9|
|     210.1|   280.5|      MSFT|          Technology|2025-02-01 00:00:00|             70.4|    Gain|  33.51|
|     120.5|   175.2|      AAPL|          Technology|2025-01-01 00:00:00|54.69999999999999|    Gain|  45.39|
|    4800.0|  7200.

In [18]:
unrealistic_df = sort_data.filter(col("Return%") > 200)

#if unrealistic_df.count() > 0:
    #unrealistic_df.write.mode("overwrite").csv(f"{error_path}/Unrealistic_Return_{timestamp}.csv")

sort_data1 = sort_data.filter(col("Return%") <= 200)
sort_data1.show()

+----------+--------+----------+--------------------+-------------------+-----------------+--------+-------+
|PriceStart|PriceEnd|     Stock|              Sector|              Month|      PriceChange|catogory|Return%|
+----------+--------+----------+--------------------+-------------------+-----------------+--------+-------+
|     180.0|   160.0|      BABA|Consumer Discreti...|2025-06-01 00:00:00|            -20.0|    Loss| -11.11|
|    2200.0|  2400.0|  RELIANCE|              Energy|2025-07-01 00:00:00|            200.0|    Gain|   9.09|
|      95.0|   105.0|       JPM|          Financials|2025-05-01 00:00:00|             10.0|    Gain|  10.53|
|    3100.0|  3500.0|       TCS|          Technology|2025-08-01 00:00:00|            400.0|    Gain|   12.9|
|     210.1|   280.5|      MSFT|          Technology|2025-02-01 00:00:00|             70.4|    Gain|  33.51|
|     120.5|   175.2|      AAPL|          Technology|2025-01-01 00:00:00|54.69999999999999|    Gain|  45.39|
|    4800.0|  7200.

In [19]:
from pyspark.sql.functions import stddev, mean
outlier_detection = sort_data1.select(mean("Return%").alias("mean"), stddev("Return%").alias("std")).collect()[0]
mean_val = outlier_detection["mean"]
std_val = outlier_detection["std"]

upper_limit = mean_val + 2 * std_val

sort_data2 = sort_data1.withColumn("Outlier",when(col("Return%") > upper_limit, "Yes").otherwise("No"))
sort_data2.show()

+----------+--------+----------+--------------------+-------------------+-----------------+--------+-------+-------+
|PriceStart|PriceEnd|     Stock|              Sector|              Month|      PriceChange|catogory|Return%|Outlier|
+----------+--------+----------+--------------------+-------------------+-----------------+--------+-------+-------+
|     180.0|   160.0|      BABA|Consumer Discreti...|2025-06-01 00:00:00|            -20.0|    Loss| -11.11|     No|
|    2200.0|  2400.0|  RELIANCE|              Energy|2025-07-01 00:00:00|            200.0|    Gain|   9.09|     No|
|      95.0|   105.0|       JPM|          Financials|2025-05-01 00:00:00|             10.0|    Gain|  10.53|     No|
|    3100.0|  3500.0|       TCS|          Technology|2025-08-01 00:00:00|            400.0|    Gain|   12.9|     No|
|     210.1|   280.5|      MSFT|          Technology|2025-02-01 00:00:00|             70.4|    Gain|  33.51|     No|
|     120.5|   175.2|      AAPL|          Technology|2025-01-01 

In [17]:
save_data = sort_data.write.mode("overwrite").option("header", True).parquet(rf"C:\Users\RakshithM\PycharmProjects\PythonProject\Miniprojects\output\output.parquet")

Py4JJavaError: An error occurred while calling o149.parquet.
: java.lang.RuntimeException: java.io.FileNotFoundException: Could not locate Hadoop executable: C:\spark\bin\winutils.exe -see https://cwiki.apache.org/confluence/display/HADOOP2/WindowsProblems
	at org.apache.hadoop.util.Shell.getWinUtilsPath(Shell.java:789)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:298)
	at org.apache.hadoop.util.Shell.getSetPermissionCommand(Shell.java:314)
	at org.apache.hadoop.fs.RawLocalFileSystem.setPermission(RawLocalFileSystem.java:1116)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkOneDirWithMode(RawLocalFileSystem.java:798)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:838)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFileSystem.java:810)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirsWithOptionalPermission(RawLocalFileSystem.java:837)
	at org.apache.hadoop.fs.RawLocalFileSystem.mkdirs(RawLocalFi